# VideoDB Indexing V2 Quickstart

Run the full flow: create video understanding outputs, index them, then search/query/aggregate over the indexed moments.

This notebook stays intentionally short. For deeper guides, see:

- `guides/indexing-v2/understanding/quickstart.ipynb`
- `guides/indexing-v2/understanding/vlm/vlm-guide.ipynb`
- `guides/indexing-v2/indexing/indexing_guide.ipynb`
- `guides/indexing-v2/indexing/custom_index.ipynb`


<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 1. Install dependencies

This installs the preview SDK branch with Understanding and Indexing V2 helpers.


In [ ]:
!pip install -q videodb python-dotenv


## 2. Connect to VideoDB

Set `VIDEO_DB_API_KEY` in Colab secrets/environment, or enter it when prompted.


In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

if not os.getenv("VIDEO_DB_API_KEY"):
    os.environ["VIDEO_DB_API_KEY"] = getpass("Enter your VideoDB API key: ")

conn = connect(api_key=os.environ["VIDEO_DB_API_KEY"])
print("Connected to VideoDB")


## 3. Choose a video

By default, this notebook uploads the sample video used in the E2E flow: **Silicon Valley - Gilfoyle is free for hire**. To use an existing video instead, comment the upload line and uncomment the `get_video` lines in the next cell.


In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=vVlEVRKv4is"  # Silicon Valley - Gilfoyle is free for hire

collection = conn.get_collection()
video = collection.upload(VIDEO_URL)

# To use an existing video instead, comment the upload line above and uncomment these lines:
# VIDEO_ID = "m-..."
# video = collection.get_video(VIDEO_ID)

print("Collection:", collection.id)
print("Video:", video.id)
video.play()


## 4. Create an Understanding run

Understanding produces reusable analyzer outputs. Here we ask for transcript, object detections, and a VLM scene description.


In [ ]:
OBJECT_LABELS = [
    "person",
    "cup",
    "bottle",
    "chair",
    "sofa",
    "diningtable",
    "laptop",
    "cell phone",
    "book",
]


In [ ]:
understanding = video.understand(
    analyzers=[
        {
            "type": "spoken_words",
            "name": "transcript",
            "config": {"language": "en"},
        },
        {
            "type": "object_detection",
            "name": "objects",
            "sampling": {"strategy": "interval", "every": 1},
            "config": {
                "labels": OBJECT_LABELS,
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        },
        {
            "type": "vlm",
            "name": "scene",
            "inputs": ["transcript", "objects"],
            "sampling": {"strategy": "uniform", "frame_count": 3},
            "config": {
                "model": "ultra",
                "prompt": (
                    "Analyze this Silicon Valley clip temporally from the sampled frame sequence. "
                    "Use the detected objects and transcript when helpful. "
                    "Describe what happens in the scene and return it in the outputs field."
                ),
                "schema": {"outputs": "text"},
            },
        },
    ],
    segmentation={"type": "shot", "threshold": 30},
)

print("Understanding ID:", understanding.id)
print("Status:", understanding.status)
understanding.list_analyzers()


## 5. Wait and fetch outputs

Each analyzer returns timestamped scene/segment outputs.


In [ ]:
understanding.wait_until_complete(timeout=3600, poll_interval=15)
print("Final status:", understanding.status)

for analyzer in understanding.list_analyzers():
    print(analyzer.name, analyzer.type, analyzer.status)


In [ ]:
def as_segments(output):
    """Handle both list outputs and {'scenes': [...]} outputs."""
    return output.get("scenes", output) if isinstance(output, dict) else output


def preview_segments(name, output, max_segments=3):
    segments = as_segments(output) or []
    print(f"{name}: {len(segments)} segments")
    print("=" * 60)
    for segment in segments[:max_segments]:
        print(f"{segment.get('start')}s → {segment.get('end')}s")
        print(segment.get("data"))
        print("-" * 60)


In [ ]:
transcript = understanding.get_analyzer("transcript").get_output()
objects = understanding.get_analyzer("objects").get_output()
scene = understanding.get_analyzer("scene").get_output()

preview_segments("transcript", transcript)
preview_segments("objects", objects)
preview_segments("scene", scene)


## 6. Create indexes

An **index** turns the fields of an understanding artifact into retrieval-ready storage. You control two things: what the index can be used for (`use_for`), and how each field is indexed (`fields`).

### `use_for` — retrieval capabilities

| Capability | Enables | Method |
|---|---|---|
| `semantic` | meaning / vector search | `semantic_search()`, `search()` |
| `query` | structured filters | `query()` |
| `aggregate` | counts, group-by, facets | `aggregate()` |

Omit `use_for` and it defaults to `["semantic", "query", "aggregate"]`. Drop `semantic` (e.g. `["query"]`) to skip embeddings — a query-only index is cheaper and ready instantly. The value `get_index()` reports back is the index's **effective** capability: `aggregate` only appears when a field is groupable, and an empty index reports `[]`.

### `fields` — how each field is indexed

`fields` maps **field groups** to the artifact fields that join them. A field's group decides what it can do:

| Group | What the field can do |
|---|---|
| `semantic` | embedded for meaning search — "find talk **about** X" |
| `fts` | full-text keyword search (stemmed, prefix) — "find where they **said** X" |
| `filter` | structured `query()` filters (exact, `contains`, ranges) |
| `aggregate` | `group_by` / counts / facets |
| `sort` | result ordering |

A field can be in several groups. Names may be **dotted paths** into nested data (`frames.detections.label`).

### Defaults — you don't have to declare everything

Omit `fields` (or any single group) and VideoDB derives it: well-known names get product defaults (recognized speech / on-screen `text` → `semantic` + `fts`; `scene_description` → `semantic`; `language` / `brand_names` → `filter` + `aggregate`), and everything else is classified by shape (prose → `semantic`; scalars → `filter` + `aggregate`, numbers also `sort`; nested objects stay stored-only). An empty group opts out: `{"filter": []}`.

### The build is asynchronous (rows-first)

Semantic indexes return as `building` and flip to `ready` once embeddings are written. Indexing is **rows-first**: records are stored before `index()` returns, so `query()` and `aggregate()` work on a `building` index right away — only `semantic_search()` waits for `ready`. Block on a build with `index.wait_until_complete()`.


In [ ]:
from datetime import datetime

run_suffix = datetime.utcnow().strftime("%Y%m%d%H%M%S")

transcript_index_name = f"transcript_{run_suffix}"
objects_index_name = f"objects_{run_suffix}"
scene_index_name = f"scene_{run_suffix}"

transcript_index = video.index(
    name=transcript_index_name,
    source=transcript
)

objects_index = video.index(
    name=objects_index_name,
    source=objects,
    use_for=["query", "aggregate"],
    fields={
        "filter": ["frames.detections.label", "frames.detections.score"],
        "aggregate": ["frames.detections.label"],
        "sort": ["frames.detections.score"],
    },
)

scene_index = video.index(
    name=scene_index_name,
    source=scene,
    use_for=["semantic", "query"],
    fields={
        "semantic": ["outputs"],
        "filter": ["outputs"],
    },
)

transcript_index, objects_index, scene_index

## 7. Wait for indexes

Search works best after indexes are ready.


In [ ]:
# Index builds are asynchronous — wait with the SDK's built-in helper.
# Query-only indexes are ready at once; semantic indexes flip from `building` to `ready`.
for index in (transcript_index, objects_index, scene_index):
    index.wait_until_complete(timeout=900, poll_interval=10)
    print(index.name, "->", index.status)
    if not index.is_successful:
        print("   build failed:", index.error)


## 8. Inspect indexed fields

Each index declares which fields can be used for semantic search, filtering, and aggregation.

In [ ]:
print("Transcript index fields:", transcript_index.fields)
print("Objects index fields:", objects_index.fields)
print("Scene index fields:", scene_index.fields)

## Choosing a retrieval method

Each retrieval method answers a different question:

| Method | Answers | How |
|---|---|---|
| `video.search(query=...)` | "find moments about X" in natural language — VideoDB plans across indexes | semantic + filters |
| `video.semantic_search(query=..., index_names=[...])` | "find talk **about** X" (meaning) on chosen semantic indexes | vector similarity |
| `video.query(index_name=..., filter=...)` | exact / structured lookup on one index's fields | field filters |
| `video.aggregate(index_name=..., group_by=...)` | counts, facets, "how many X" | group-by |
| `video.ask(query=...)` | a synthesized answer with cited sources | retrieval + LLM |

A field must be in the `filter` group to appear in a `query()` filter, and in `aggregate` to be a `group_by` — inspect the mapping with `index.field_schema`. `query()`/`aggregate()` target a **single** index; `semantic_search()` can span several via `index_names`.


## 9. Search naturally

Use `video.search(...)` for broad natural-language moment retrieval.

In [ ]:
results = video.search(
    query="a gift arrives at the front door",
    top_k=5,
    mode="default",
    return_fields="all",
)

if results:
    results[0].play()
else:
    print("No results found")

## 10. Search a specific semantic index

Use `video.semantic_search(...)` when you want semantic retrieval over a known index.

In [ ]:
results = video.semantic_search(
    query="a man drops an electronic device in a trash bin",
    index_names=[scene_index_name],
    top_k=5,
    score_threshold=0.2,
    return_fields="all",
)

if results:
    results[0].play()
else:
    print("No results found")

## 11. Ask questions with sources

Use `video.ask(...)` when you want an answer grounded in retrieved video moments.

In [ ]:
answer = video.ask(
    question="what does the character say after he changes his linkedin status",
    top_k=15,
    mode="default",
    include_sources=True,
)

print(answer.answer)
answer.sources[0].play()

## 12. Query exact scene text with AND / OR filters

Use `video.query(...)` when you know the exact indexed field/filter.

In [ ]:
delivery_query_results = video.query(
    index_name=scene_index_name,
    filter=[{"field": "outputs", "op": "contains", "value": "front door"}],
    limit=10,
    return_fields="all",
)

delivery_query_results

In [ ]:
trash_where = {
    "and": [
        {
            "or": [
                {"field": "outputs", "op": "contains", "value": "dustbin"},
                {"field": "outputs", "op": "contains", "value": "trash"},
            ]
        },
        {
            "or": [
                {"field": "outputs", "op": "contains", "value": "white cord"},
                {"field": "outputs", "op": "contains", "value": "wired device"},
                {"field": "outputs", "op": "contains", "value": "electronic device"},
                {"field": "outputs", "op": "contains", "value": "USB missile launcher"},
            ]
        },
    ]
}

trash_query_results = video.query(
    index_name=scene_index_name,
    filter=trash_where,
    limit=10,
    return_fields="all",
)

trash_query_results

## 13. Query object detections

The object index exposes detection labels and scores for structured lookup.

In [ ]:
cup_results = video.query(
    index_name=objects_index_name,
    filter=[{"field": "frames.detections.label", "op": "contains", "value": "cup"}],
    limit=10,
    return_fields="all",
)

cup_results

## 14. Aggregate indexed fields

Use `video.aggregate(...)` for counts, groups, and facets.

In [ ]:
object_counts = video.aggregate(
    index_name=objects_index_name,
    group_by="frames.detections.label",
    metric="count",
    limit=20,
)

gadget_object_counts = video.aggregate(
    index_name=objects_index_name,
    filter={
        "or": [
            {"field": "frames.detections.label", "op": "contains", "value": "laptop"},
            {"field": "frames.detections.label", "op": "contains", "value": "cell phone"},
            {"field": "frames.detections.label", "op": "contains", "value": "cup"},
            {"field": "frames.detections.label", "op": "contains", "value": "bottle"},
            {"field": "frames.detections.label", "op": "contains", "value": "book"},
        ]
    },
    group_by="frames.detections.label",
    metric="count",
    limit=10,
)

print("Object counts:", object_counts)
print("Selected object counts:", gadget_object_counts)

## 15. Optional cleanup

Only run deletion when you are done with these preview records.

In [ ]:
DELETE_INDEXES = False

if DELETE_INDEXES:
    video.delete_index(index_id=transcript_index.index_id)
    video.delete_index(index_id=objects_index.index_id)
    video.delete_index(index_id=scene_index.index_id)
    print("Deleted indexes")
else:
    print("Skipping delete. Set DELETE_INDEXES=True to delete these indexes.")

In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    understanding.delete()
    print("Deleted", understanding.id)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")